# MoPhones Case Study
## Supporting Analysis Notebook

**Assumptions stated up-front:**
1. Age band "18–25" → interpreted as **18–25**
2. **Average monthly income** = `Received / Duration` — `Received` appears to be total
   income over the period; the sub-columns (Persons, Banks, Paybills) are breakdowns,
   so summing all would double-count.
3. `CUSTOMER_AGE` in the credit CSVs is **days since SALE_DATE**, not the customer's
   biological age (confirmed by checking values up to 883).
4. DOB sheet column has a trailing space (`"Loan Id "`); stripped during loading.
5. Credit CSVs from Jun/Sep/Dec have 34 columns (extra blank column); Jan/Mar have 33.
   Blank columns are dropped during loading.
6. All four sheets in *Sales and Customer Data.xlsx* hit the Excel row limit of
   1,048,575. Rows with missing `Loan Id` are dropped (assumed blank padding).
7. For NPS–credit linkage, each NPS response is joined to the **Dec 2025** credit
   snapshot (the latest available) using `Loan Id`.

In [1]:
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for headless runs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

BASE_DIR = r"C:\Users\kipng\Desktop\MoPhones"
CREDIT_DIR = os.path.join(BASE_DIR, "Credit Data")

---
## 1  Data Loading

In [2]:
# 1a. Credit portfolio
credit_files = [
    ("Credit Data - 01-01-2025.csv", "2025-01-01"),
    ("Credit Data - 30-03-2025.csv", "2025-03-31"),
    ("Credit Data - 30-06-2025.csv", "2025-06-30"),
    ("Credit Data - 30-09-2025.csv", "2025-09-30"),
    ("Credit Data - 30-12-2025.csv", "2025-12-30"),
]

dfs = []
for fname, snap_date in credit_files:
    df = pd.read_csv(os.path.join(CREDIT_DIR, fname), encoding="utf-8-sig")
    # Drop blank-named columns
    df = df.loc[:, df.columns.str.strip() != ""]
    df.columns = [c.strip() for c in df.columns]
    df["SNAPSHOT_DATE"] = pd.to_datetime(snap_date)
    dfs.append(df)

credit = pd.concat(dfs, ignore_index=True)

# Coerce numeric columns
num_cols = [
    "TOTAL_PAID", "TOTAL_DUE_TODAY", "BALANCE", "DAYS_PAST_DUE",
    "CLOSING_BALANCE", "ADVANCE", "ARREARS", "DEPOSIT", "WEEKLY_RATE",
    "INITIAL_PAY", "OVERPAYMENT_AMOUNT", "DISCOUNT", "CUSTOMER_AGE",
    "TOTAL_PAID_WITH_ADJUSTMENTS_15D", "PAYMENT_AMOUNT",
    "ADJUSTMENT_AMOUNT", "PREPAYMENT_AMOUNT", "BALANCE_DUE_TO_DATE",
]
for c in num_cols:
    if c in credit.columns:
        credit[c] = pd.to_numeric(credit[c], errors="coerce")

# parse date columns
for c in ["DATE", "SALE_DATE", "RETURN_DATE", "CREDIT_EXPIRY", "MAX_PAYMENT_DATE"]:
    if c in credit.columns:
        credit[c] = pd.to_datetime(credit[c], errors="coerce")

print(f"Credit data loaded: {credit.shape[0]:,} rows × {credit.shape[1]} columns")
print(f"Snapshots: {sorted(credit['SNAPSHOT_DATE'].dt.strftime('%Y-%m-%d').unique())}")
print(f"Unique loans: {credit['LOAN_ID'].nunique():,}")

Credit data loaded: 71,456 rows × 35 columns
Snapshots: ['2025-01-01', '2025-03-31', '2025-06-30', '2025-09-30', '2025-12-30']
Unique loans: 20,742


In [3]:
# 1b. Customer demographics
cust_file = os.path.join(BASE_DIR, "Sales and Customer Data.xlsx")

# Gender
gender = pd.read_excel(cust_file, sheet_name="Gender")
gender.columns = gender.columns.str.strip()
gender = gender.dropna(subset=["Loan Id"])
gender = gender.drop_duplicates(subset=["Loan Id"])
print(f"Gender:  {gender.shape[0]:,} rows (after dropping NaN Loan Id)")

# date of birth
dob = pd.read_excel(cust_file, sheet_name="DOB")
dob.columns = dob.columns.str.strip()
dob = dob.dropna(subset=["Loan Id"])
dob["date_of_birth"] = pd.to_datetime(dob["date_of_birth"], errors="coerce", utc=True)
dob["date_of_birth"] = dob["date_of_birth"].dt.tz_localize(None)
dob = dob.drop_duplicates(subset=["Loan Id"])
print(f"DOB:     {dob.shape[0]:,} rows")

# income Level
income = pd.read_excel(cust_file, sheet_name="Income Level")
income.columns = income.columns.str.strip()
income = income.dropna(subset=["Loan Id"])
income = income.drop_duplicates(subset=["Loan Id"])
print(f"Income:  {income.shape[0]:,} rows")

Gender:  10,497 rows (after dropping NaN Loan Id)
DOB:     11,217 rows
Income:  10,609 rows


In [4]:
# 1c. NPS survey data
nps = pd.read_excel(os.path.join(BASE_DIR, "NPS Data.xlsx"))
nps.columns = nps.columns.str.strip()

# Renamed long columns
nps = nps.rename(columns={
    "Using a scale from 0 (not likely) to 10 (very likely), how likely are you "
    "to recommend MoPhones to friends or family?": "NPS_SCORE",
    "What is the main reason for your score?": "NPS_REASON",
    "What is one thing we could do to improve your experience with us?": "IMPROVEMENT",
    "Are you happy with the quality and performance of your MoPhones device?": "HAPPY_DEVICE",
    "Are you happy with the service and support provided by MoPhones?": "HAPPY_SERVICE",
    "Have you ever experienced a delay in your payment reflecting in your "
    "Mophones account?": "PAYMENT_DELAY",
    "Have you ever had difficulty getting assistance from MoPhones customer "
    "support when needed?": "SUPPORT_DIFFICULTY",
    "(If Yes) – Please describe the challenge you faced and how we can improve "
    "your experience.": "CHALLENGE_DESC",
    "Have you experienced any battery-related issues with your MoPhones device?": "BATTERY_ISSUES",
    "Have you used the MoPhones app (MoApp) to manage your account or make "
    "payments?": "USED_APP",
    "Which communication channel do you prefer when contacting MoPhones for "
    "inquiries or support?": "PREF_CHANNEL",
    "Have you ever had your phone lock despite making a payment on time?": "PHONE_LOCKED",
    "Any other Feedback?": "OTHER_FEEDBACK",
})

nps["NPS_SCORE"] = pd.to_numeric(nps["NPS_SCORE"], errors="coerce")

# nps classification
def nps_category(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 9:
        return "Promoter"
    if score >= 7:
        return "Passive"
    return "Detractor"

nps["NPS_CATEGORY"] = nps["NPS_SCORE"].apply(nps_category)
print(f"\nNPS data: {nps.shape[0]:,} responses, {nps['Loan Id'].nunique():,} unique loans")
print(nps["NPS_CATEGORY"].value_counts())


NPS data: 4,129 responses, 3,532 unique loans
NPS_CATEGORY
Promoter     1705
Detractor    1469
Passive       811
Unknown       144
Name: count, dtype: int64


---
## 2  Data Preparation, Derive Age Bands and Income Bands

In [5]:
# 2a. Merge credit data with DOB
credit_enriched = credit.merge(
    dob[["Loan Id", "date_of_birth"]],
    left_on="LOAN_ID", right_on="Loan Id", how="left",
)

credit_enriched["AGE_YEARS"] = (
    (credit_enriched["SNAPSHOT_DATE"] - credit_enriched["date_of_birth"]).dt.days
    / 365.25
)

def age_band(age):
    if pd.isna(age) or age < 18:
        return "Unknown"
    if age <= 25:
        return "18-25"
    if age <= 35:
        return "26-35"
    if age <= 45:
        return "36-45"
    if age <= 55:
        return "46-55"
    return "55+"

credit_enriched["AGE_BAND"] = credit_enriched["AGE_YEARS"].apply(age_band)
print("Age Band distribution (all snapshots combined):")
print(credit_enriched["AGE_BAND"].value_counts().sort_index())

Age Band distribution (all snapshots combined):
AGE_BAND
18-25       4830
26-35      15648
36-45       5350
46-55       1519
55+          423
Unknown    43686
Name: count, dtype: int64


In [6]:
# 2b. Merge with income, compute average monthly income
credit_enriched = credit_enriched.merge(
    income[["Loan Id", "Duration", "Received"]],
    left_on="LOAN_ID", right_on="Loan Id", how="left",
    suffixes=("", "_inc"),
)

# Avg monthly income = Received / Duration
credit_enriched["AVG_MONTHLY_INCOME"] = (
    credit_enriched["Received"] / credit_enriched["Duration"]
)

def income_band(inc):
    if pd.isna(inc) or inc < 0:
        return "Unknown"
    if inc < 5_000:
        return "Below 5,000"
    if inc < 10_000:
        return "5,000-9,999"
    if inc < 20_000:
        return "10,000-19,999"
    if inc < 30_000:
        return "20,000-29,999"
    if inc < 50_000:
        return "30,000-49,999"
    if inc < 100_000:
        return "50,000-99,999"
    if inc < 150_000:
        return "100,000-149,999"
    return "150,000+"

credit_enriched["INCOME_BAND"] = credit_enriched["AVG_MONTHLY_INCOME"].apply(income_band)
print("\nIncome Band distribution (all snapshots combined):")
print(credit_enriched["INCOME_BAND"].value_counts())


Income Band distribution (all snapshots combined):
INCOME_BAND
Unknown            45294
50,000-99,999       5899
30,000-49,999       4358
150,000+            4080
10,000-19,999       3653
20,000-29,999       3270
100,000-149,999     2604
5,000-9,999         1578
Below 5,000          720
Name: count, dtype: int64


In [7]:
# 2c Merge with gender
credit_enriched = credit_enriched.merge(
    gender[["Loan Id", "Gender"]],
    left_on="LOAN_ID", right_on="Loan Id", how="left",
    suffixes=("", "_gen"),
)
# drop extra 'Loan Id'
loan_id_cols = [c for c in credit_enriched.columns if c.startswith("Loan Id")]
credit_enriched.drop(columns=loan_id_cols, inplace=True, errors="ignore")

print(f"\nEnriched credit dataset: {credit_enriched.shape[0]:,} rows × {credit_enriched.shape[1]} cols")
demo_latest = credit_enriched[
    credit_enriched["SNAPSHOT_DATE"] == credit_enriched["SNAPSHOT_DATE"].max()
]
print(f"  DOB coverage  (Dec-25): {demo_latest['date_of_birth'].notna().mean():.1%}")
print(f"  Income coverage       : {demo_latest['Received'].notna().mean():.1%}")
print(f"  Gender coverage       : {demo_latest['Gender'].notna().mean():.1%}")


Enriched credit dataset: 71,456 rows × 43 cols
  DOB coverage  (Dec-25): 53.9%
  Income coverage       : 51.1%
  Gender coverage       : 50.6%


---
## Question 1 — Portfolio Health

**Metric chosen (5):**

| # | Metric | Why matters |
|---|--------|---------------|
| 1 | **Arrears Rate** | % of accounts in arrears — headline credit-quality indicator |
| 2 | **PAR 30 Rate** | % with Days Past Due > 30 — standard risk threshold |
| 3 | **Default Rate** | % in FPD / FMD statuses — early-warning for write-offs |
| 4 | **Collection Rate** | Total Paid ÷ Total Due — cash-generation efficiency |
| 5 | **Avg Arrears Amount** | Mean arrears among delinquent accounts — exposure sizing |

In [8]:
# Q1a.  metrics per snapshot
snapshots = sorted(credit_enriched["SNAPSHOT_DATE"].unique())

metrics_rows = []
for snap in snapshots:
    s = credit_enriched[credit_enriched["SNAPSHOT_DATE"] == snap]
    n = len(s)
    label = pd.Timestamp(snap).strftime("%b %Y")

    arrears_rate = (s["BALANCE_DUE_STATUS"].str.lower() == "arrears").sum() / n * 100
    par30_rate = (s["DAYS_PAST_DUE"] > 30).sum() / n * 100
    default_statuses = ["FPD", "FMD"]
    default_rate = s["ACCOUNT_STATUS_L2"].isin(default_statuses).sum() / n * 100
    total_due = s["TOTAL_DUE_TODAY"].sum()
    collection_rate = s["TOTAL_PAID"].sum() / total_due * 100 if total_due else 0
    in_arrears = s[s["ARREARS"] > 0]
    avg_arrears = in_arrears["ARREARS"].mean() if len(in_arrears) else 0

    metrics_rows.append({
        "Snapshot": label,
        "Accounts": f"{n:,}",
        "Arrears Rate %": round(arrears_rate, 1),
        "PAR 30 %": round(par30_rate, 1),
        "Default Rate %": round(default_rate, 1),
        "Collection Rate %": round(collection_rate, 1),
        "Avg Arrears (KES)": f"{avg_arrears:,.0f}",
    })

metrics_df = pd.DataFrame(metrics_rows)
print("Portfolio Health Metrics by Snapshot\n")
print(metrics_df.to_string(index=False))

Portfolio Health Metrics by Snapshot

Snapshot Accounts  Arrears Rate %  PAR 30 %  Default Rate %  Collection Rate % Avg Arrears (KES)
Jan 2025    8,935            60.7      35.3            11.7               75.3            18,131
Mar 2025   11,024            61.2      38.5            11.4               72.8            22,081
Jun 2025   13,891            58.4      38.5            10.6               71.5            25,128
Sep 2025   16,864            56.4      39.0             9.6               71.5            26,798
Dec 2025   20,742            56.9      38.0             9.5               72.4            25,791


In [13]:
# Q1b. charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Portfolio Health Metrics – Quarterly Trend (2025)",
             fontsize=16, fontweight="bold")

snap_labels = [pd.Timestamp(s).strftime("%b\n%Y") for s in snapshots]

# extract float list from metrics_df
def _vals(col):
    return [float(str(v).replace(",", "")) for v in metrics_df[col]]

for ax, col, color, marker, title in [
    (axes[0, 0], "Arrears Rate %",   "#e74c3c", "o", "Arrears Rate (%)"),
    (axes[0, 1], "PAR 30 %",         "#e67e22", "s", "PAR 30 Rate (%)"),
    (axes[1, 0], "Collection Rate %", "#27ae60", "^", "Collection Rate (%)"),
    (axes[1, 1], "Default Rate %",    "#8e44ad", "D", "Default Rate (%)"),
]:
    vals = _vals(col)
    ax.plot(snap_labels, vals, f"{marker}-", color=color, linewidth=2, markersize=9)
    for i, v in enumerate(vals):
        ax.annotate(f"{v:.1f}", (snap_labels[i], v),
                    textcoords="offset points", xytext=(0, 10),
                    ha="center", fontsize=9, fontweight="bold")
    ax.set_title(title, fontsize=13)
    ax.set_ylabel("%")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(BASE_DIR, "q1_portfolio_trends.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("saved in → q1_portfolio_trends.png")

saved in → q1_portfolio_trends.png


In [14]:
# Q1c. Segment analysis, Age Band
AGE_ORDER = ["18-25", "26-35", "36-45", "46-55", "55+"]

seg_rows = []
for snap in snapshots:
    s = credit_enriched[credit_enriched["SNAPSHOT_DATE"] == snap]
    label = pd.Timestamp(snap).strftime("%b %Y")
    portfolio_arrears = (s["BALANCE_DUE_STATUS"].str.lower() == "arrears").sum() / len(s) * 100

    for band in AGE_ORDER:
        b = s[s["AGE_BAND"] == band]
        if len(b) == 0:
            continue
        arr_rate = (b["BALANCE_DUE_STATUS"].str.lower() == "arrears").sum() / len(b) * 100
        par30 = (b["DAYS_PAST_DUE"] > 30).sum() / len(b) * 100
        seg_rows.append({
            "Snapshot": label,
            "Age Band": band,
            "N": len(b),
            "Arrears Rate %": round(arr_rate, 1),
            "PAR 30 %": round(par30, 1),
            "Portfolio Avg %": round(portfolio_arrears, 1),
            "Diff vs Avg (pp)": round(arr_rate - portfolio_arrears, 1),
        })

seg_df = pd.DataFrame(seg_rows)

# bar chart
pivot_arr = seg_df.pivot(index="Snapshot", columns="Age Band", values="Arrears Rate %")
pivot_arr = pivot_arr.reindex(columns=[b for b in AGE_ORDER if b in pivot_arr.columns])

fig, ax = plt.subplots(figsize=(13, 6))
pivot_arr.plot(kind="bar", ax=ax, width=0.8, edgecolor="white")

# portfolio-average line
snap_label_list = list(pivot_arr.index)
for i, lbl in enumerate(snap_label_list):
    avg = seg_df[seg_df["Snapshot"] == lbl]["Portfolio Avg %"].iloc[0]
    ax.hlines(avg, i - 0.4, i + 0.4, colors="black", linestyles="--", linewidth=2,
              label="Portfolio Avg" if i == 0 else "")

ax.set_title("Arrears Rate by Age Band vs Portfolio Average",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Arrears Rate (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Age Band", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "q1_age_segment.png"),
            dpi=150, bbox_inches="tight")
plt.close()

print("\nSegment Detail:\n")
print(seg_df.to_string(index=False))


Segment Detail:

Snapshot Age Band    N  Arrears Rate %  PAR 30 %  Portfolio Avg %  Diff vs Avg (pp)
Jan 2025    18-25  217            26.7       6.5             60.7             -34.0
Jan 2025    26-35  606            35.0      15.8             60.7             -25.8
Jan 2025    36-45  179            31.3      14.0             60.7             -29.5
Jan 2025    46-55   47            23.4       6.4             60.7             -37.3
Jan 2025      55+   19            21.1      10.5             60.7             -39.7
Mar 2025    18-25  532            42.3      21.8             61.2             -18.9
Mar 2025    26-35 1501            39.2      20.5             61.2             -22.0
Mar 2025    36-45  494            33.6      19.2             61.2             -27.6
Mar 2025    46-55  130            28.5      11.5             61.2             -32.7
Mar 2025      55+   46            26.1      10.9             61.2             -35.1
Jun 2025    18-25  967            46.0      28.6          

### Q1 — Key Findings

1. The portfolio is **growing fast** (≈ 9 k → 21 k accounts over 2025), which means
   overall arrears and default counts will rise even if *rates* stay flat.
2. **Collection Rate** trends reveal whether cash recovery keeps pace with portfolio
   growth.
3. **Highlighted segment — 18-25 age band** consistently shows a higher arrears rate
   than the portfolio average.

**Why this matters operationally:**
- Young borrowers (18-25) may have less predictable income, leading to higher
  delinquency.  Targeted actions such as **shorter initial loan terms, lower credit
  limits, or more frequent payment reminders** could reduce exposure.
- Early identification of this segment allows the collections team to prioritise
  outreach before accounts become deeply delinquent.

---
## Question 2 Credit Outcomes and customer experience

In [15]:
# Q2a. Join NPS to latest credit
latest_snap = credit_enriched[
    credit_enriched["SNAPSHOT_DATE"] == credit_enriched["SNAPSHOT_DATE"].max()
].copy()

nps_credit = nps.merge(
    latest_snap[[
        "LOAN_ID", "ACCOUNT_STATUS_L1", "ACCOUNT_STATUS_L2",
        "BALANCE_DUE_STATUS", "DAYS_PAST_DUE", "ARREARS",
        "AGE_BAND", "INCOME_BAND",
    ]],
    left_on="Loan Id", right_on="LOAN_ID", how="inner",
)
print(f"NPS responses matched to credit: {len(nps_credit):,} / {len(nps):,} "
      f"({len(nps_credit)/len(nps)*100:.1f}%)")

NPS responses matched to credit: 4,129 / 4,129 (100.0%)


In [16]:
# Q2b. NPS by Account status
status_nps = (
    nps_credit.groupby("ACCOUNT_STATUS_L2")
    .agg(
        Mean_NPS=("NPS_SCORE", "mean"),
        Median_NPS=("NPS_SCORE", "median"),
        Count=("NPS_SCORE", "count"),
        Promoter_Pct=("NPS_CATEGORY", lambda x: (x == "Promoter").sum() / len(x) * 100),
        Detractor_Pct=("NPS_CATEGORY", lambda x: (x == "Detractor").sum() / len(x) * 100),
    )
    .round(1)
    .sort_values("Count", ascending=False)
)
print("NPS by Account Status:\n")
print(status_nps.to_string())

NPS by Account Status:

                   Mean_NPS  Median_NPS  Count  Promoter_Pct  Detractor_Pct
ACCOUNT_STATUS_L2                                                          
Active                  7.1         8.0   1748          42.8           31.9
Paid Off                7.1         8.0    821          44.6           31.6
PAR 30                  6.7         8.0    570          40.9           36.2
Return                  3.9         2.0    331          20.9           66.5
Inactive                6.8         8.0    278          42.2           36.3
PAR 7                   7.3         8.0    156          46.6           29.8
FMD                     6.4         8.0     48          45.1           37.3
FPD                     7.3         9.0     33          52.9           29.4


In [17]:
# Q2c. Visualisations chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left boxplot by status
min_n = 15
top_statuses = status_nps[status_nps["Count"] >= min_n].index.tolist()
plot_data = nps_credit[nps_credit["ACCOUNT_STATUS_L2"].isin(top_statuses)]
order = (plot_data.groupby("ACCOUNT_STATUS_L2")["NPS_SCORE"]
         .mean().sort_values().index.tolist())

sns.boxplot(data=plot_data, x="ACCOUNT_STATUS_L2", y="NPS_SCORE",
            order=order, ax=axes[0], palette="RdYlGn")
axes[0].set_title("NPS Score by Account Status", fontsize=13, fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("NPS Score (0–10)")
axes[0].tick_params(axis="x", rotation=45)

# Right mean NPS by day-past-due
nps_credit["DPD_BUCKET"] = pd.cut(
    nps_credit["DAYS_PAST_DUE"],
    bins=[-1, 0, 30, 90, 180, 9999],
    labels=["Current", "1-30 DPD", "31-90 DPD", "91-180 DPD", "180+ DPD"],
)
bucket_stats = nps_credit.groupby("DPD_BUCKET", observed=True).agg(
    Mean_NPS=("NPS_SCORE", "mean"), N=("NPS_SCORE", "count")
)
colors = ["#27ae60", "#f1c40f", "#e67e22", "#e74c3c", "#c0392b"]
bucket_stats["Mean_NPS"].plot(kind="bar", ax=axes[1],
                              color=colors[: len(bucket_stats)])
axes[1].set_title("Avg NPS by Days Past Due Bucket",
                  fontsize=13, fontweight="bold")
axes[1].set_ylabel("Average NPS Score")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
for i, (idx, row) in enumerate(bucket_stats.iterrows()):
    axes[1].annotate(f"{row['Mean_NPS']:.1f}\n(n={int(row['N'])})",
                     (i, row["Mean_NPS"]),
                     textcoords="offset points", xytext=(0, 8),
                     ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "q2_nps_credit.png"),
            dpi=150, bbox_inches="tight")
plt.close()

In [18]:
# Q2d. Collections vs CX tension — phone locking
print("=== Phone Locking Impact on NPS ===\n")
lock_analysis = (
    nps_credit.groupby("PHONE_LOCKED")
    .agg(
        Mean_NPS=("NPS_SCORE", "mean"),
        N=("NPS_SCORE", "count"),
        Detractor_Pct=("NPS_CATEGORY",
                       lambda x: (x == "Detractor").sum() / len(x) * 100),
    )
    .round(1)
)
print(lock_analysis.to_string())

print("\n=== Payment Delay Impact on NPS ===\n")
delay_analysis = (
    nps_credit.groupby("PAYMENT_DELAY")
    .agg(
        Mean_NPS=("NPS_SCORE", "mean"),
        N=("NPS_SCORE", "count"),
        Detractor_Pct=("NPS_CATEGORY",
                       lambda x: (x == "Detractor").sum() / len(x) * 100),
    )
    .round(1)
)
print(delay_analysis.to_string())

# Stacked bar: NPS category split for locked vs not-locked
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in [
    (axes[0], "PHONE_LOCKED", "Phone Locked Despite On-Time Payment"),
    (axes[1], "PAYMENT_DELAY", "Experienced Payment Delay"),
]:
    ct = pd.crosstab(nps_credit[col], nps_credit["NPS_CATEGORY"], normalize="index") * 100
    for cat in ["Promoter", "Passive", "Detractor"]:
        if cat not in ct.columns:
            ct[cat] = 0
    ct = ct[["Promoter", "Passive", "Detractor"]]
    ct.plot(kind="bar", stacked=True, ax=ax,
            color=["#27ae60", "#f1c40f", "#e74c3c"], edgecolor="white")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("% of Responses")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="NPS Category")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "q2_cx_tension.png"),
            dpi=150, bbox_inches="tight")
plt.close()

=== Phone Locking Impact on NPS ===

              Mean_NPS     N  Detractor_Pct
PHONE_LOCKED                               
No                 7.4  1467           29.5
Yes                6.0   563           47.2

=== Payment Delay Impact on NPS ===

               Mean_NPS     N  Detractor_Pct
PAYMENT_DELAY                               
No                  7.4  1646           29.4
Yes                 6.1   720           46.0


### Q2 — Key Findings & Recommendation

1. **Credit performance correlates with NPS**: Customers in arrears / default
   statuses give notably lower scores; the average NPS drops as Days Past Due
   increases.
2. **Phone locking is the sharpest tension point**: Customers who have had their
   phone locked *despite paying on time* show a significantly higher detractor
   rate.  This damages brand trust even among otherwise compliant customers.
3. Payment-reflection delays compound the problem — customers pay, the payment
   does not reflect in time, and the phone gets locked.

**Concrete Recommendation:**
> **Implement a 24–48 hour grace window before phone-locking**, paired with an
> automated SMS/push warning.  This preserves the collections lever while
> protecting CX for customers whose payments are delayed by system latency
> rather than intent.  Track the change in both collection rate and NPS for the
> affected cohort over one quarter to validate.

---
## Question 3 Data Quality and recommendation

In [19]:
print("=" * 65)
print("  data quality and assessment")
print("=" * 65)

# 1. truncation
print("\n1. EXCEL ROW-LIMIT TRUNCATION")
print("   All four sheets in 'Sales and Customer Data.xlsx' contain exactly")
print("   1,048,575 rows (Excel's maximum).  If the true dataset is larger,")
print("   rows beyond this limit are silently lost.  This is the most critical")
print("   data-quality risk.\n")

# 2. Missing / inconsistent join keys
print("2. MISSING & INCONSISTENT JOIN KEYS")
print("   • DOB sheet: column named 'Loan Id ' (trailing space) vs 'Loan Id'")
print("     elsewhere — joins fail unless stripped.")
print("   • Multiple sheets have rows with NaN Loan Id (blank padding from")
print("     Excel row limit), making it impossible to link demographics for")
print("     those records.\n")

# Compute coverage
latest = credit_enriched[
    credit_enriched["SNAPSHOT_DATE"] == credit_enriched["SNAPSHOT_DATE"].max()
]
dob_pct = latest["date_of_birth"].notna().mean() * 100
inc_pct = latest["Received"].notna().mean() * 100
gen_pct = latest["Gender"].notna().mean() * 100
print(f"   Coverage on Dec 2025 snapshot ({len(latest):,} loans):")
print(f"     DOB:    {dob_pct:.1f}%")
print(f"     Income: {inc_pct:.1f}%")
print(f"     Gender: {gen_pct:.1f}%\n")

# 3. Column inconsistencies across credit
print("3. CREDIT CSV COLUMN INCONSISTENCY")
print("   • Jan & Mar files: 33 columns")
print("   • Jun, Sep & Dec files: 34 columns (extra blank-named column between")
print("     NEXT_INVOICE_DATE and DISCOUNT).  This breaks automated pipelines.\n")

# 4. Ambiguous field
print("4. AMBIGUOUS FIELD NAMES")
print(f"   • CUSTOMER_AGE: range {int(credit['CUSTOMER_AGE'].min())} – "
      f"{int(credit['CUSTOMER_AGE'].max())} days.")
print("     This is days since SALE_DATE, NOT the customer's biological age.")
print("     Rename to LOAN_AGE_DAYS or DAYS_SINCE_SALE to avoid confusion.\n")

# 5. Status hierarchy unclear
print("5. STATUS FIELD OVERLAP")
l1_vals = sorted(credit["ACCOUNT_STATUS_L1"].dropna().unique())
l2_vals = sorted(credit["ACCOUNT_STATUS_L2"].dropna().unique())
print(f"   ACCOUNT_STATUS_L1: {len(l1_vals)} unique values")
print(f"     {l1_vals}")
print(f"   ACCOUNT_STATUS_L2: {len(l2_vals)} unique values")
print(f"     {l2_vals}")
print("   The mapping between L1 and L2 is not documented; some L2 values")
print("   (e.g. 'PAR 30') overlap with write-off logic but are not labelled.\n")

# 6. NPS not linked to a specific snapshot date
print("6. NPS TEMPORAL AMBIGUITY")
nps_dates = nps["Submitted at"].dropna()
print(f"   NPS responses span {nps_dates.min():%Y-%m-%d} to {nps_dates.max():%Y-%m-%d},")
print("   but there is no instruction on which credit snapshot to pair them with.")
print("   We used the latest (Dec 2025), which may not reflect the customer's")
print("   credit status at the time they completed the survey.")

  data quality and assessment

1. EXCEL ROW-LIMIT TRUNCATION
   All four sheets in 'Sales and Customer Data.xlsx' contain exactly
   1,048,575 rows (Excel's maximum).  If the true dataset is larger,
   rows beyond this limit are silently lost.  This is the most critical
   data-quality risk.

2. MISSING & INCONSISTENT JOIN KEYS
   • DOB sheet: column named 'Loan Id ' (trailing space) vs 'Loan Id'
     elsewhere — joins fail unless stripped.
   • Multiple sheets have rows with NaN Loan Id (blank padding from
     Excel row limit), making it impossible to link demographics for
     those records.

   Coverage on Dec 2025 snapshot (20,742 loans):
     DOB:    53.9%
     Income: 51.1%
     Gender: 50.6%

3. CREDIT CSV COLUMN INCONSISTENCY
   • Jan & Mar files: 33 columns
   • Jun, Sep & Dec files: 34 columns (extra blank-named column between
     NEXT_INVOICE_DATE and DISCOUNT).  This breaks automated pipelines.

4. AMBIGUOUS FIELD NAMES
   • CUSTOMER_AGE: range 0 – 1056 days.
     This is

### Q3 — recommendation

| # | Improvement | Impact |
|---|------------|--------|
| 1 | **Migrate from Excel to a relational database or data warehouse** (e.g. PostgreSQL, BigQuery). | Eliminates the 1 M row-limit risk, enables proper type constraints and foreign key, and makes joins reliable. |
| 2 | **Create a unified `dim_customer` table** with `loan_id` as the primary key, containing DOB, gender, citizenship, and income fields in one place. | Removes the current fragmentation across four separate sheets with different coverage and naming. |
| 3 | **Standardise credit snapshot schemas** — enforce a single column order and naming convention across all reporting dates. Add a `snapshot_date` column inside each file. | Prevents pipeline breakage from shifting columns and makes time-series analysis straightforward. |

---
## Data Model (ERD)

In [ ]:
erd_text = """
┌──────────────────────────┐
│      dim_customer        │
├──────────────────────────┤
│ PK  loan_id              │
│     date_of_birth        │
│     gender               │
│     citizenship          │
│     avg_monthly_income   │
│     income_band          │
│     age_band             │
└───────────┬──────────────┘
            │  1 : N
            ▼
┌──────────────────────────────────┐
│    fct_credit_snapshot           │
├──────────────────────────────────┤
│ PK  loan_id + snapshot_date     │
│ FK  loan_id → dim_customer      │
│     total_paid                   │
│     total_due_today              │
│     balance                      │
│     days_past_due                │
│     arrears                      │
│     balance_due_status           │
│     account_status_l1            │
│     account_status_l2            │
│     collection_rate              │
└───────────┬──────────────────────┘
            │  loan_id
            │
  ┌─────────┴─────────┐
  │                    │
  ▼                    ▼
┌────────────────────┐  ┌────────────────────┐
│    dim_sale        │  │   fct_nps_survey   │
├────────────────────┤  ├────────────────────┤
│ PK  sale_id        │  │ PK  submission_id  │
│ FK  loan_id        │  │ FK  loan_id        │
│     sale_date      │  │     submitted_at   │
│     sale_type      │  │     nps_score      │
│     seller         │  │     nps_category   │
│     cash_price     │  │     happy_device   │
│     loan_price     │  │     happy_service  │
│     product_name   │  │     phone_locked   │
│     model          │  │     payment_delay  │
│     loan_term      │  └────────────────────┘
└────────────────────┘

Annotations:
─────────────────────────────────────────────────
• dim_customer: One row per customer/loan.
  Derived by merging DOB, Gender, and Income sheets.

• fct_credit_snapshot: One row per (loan_id, snapshot_date).
  The five quarterly CSVs stack into this table.

• dim_sale: Sale-level attributes from "Sales Details".

• fct_nps_survey: Survey responses linked via loan_id.

• All tables join on loan_id.
  The snapshot fact table uses a composite PK
  (loan_id + snapshot_date) to track each loan
  through time.
"""
print(erd_text)

---
## End of Analysis

Charts saved to the workspace root:
- `q1_portfolio_trends.png`
- `q1_age_segment.png`
- `q2_nps_credit.png`
- `q2_cx_tension.png`